# Healthcare Claims & Utilization Analytics Pipeline

This notebook builds a medallion-style healthcare analytics pipeline using PySpark, SQL, and Delta tables. Raw Synthea healthcare data is validated and transformed into Silver tables, then aggregated into Gold tables for downstream business intelligence and Power BI reporting.

**Analysis Period:** 2010–2019

## 1. Silver Layer — Patients

Loads the raw patient dataset, validates data quality, selects analytics-relevant fields, and creates patient age and age-group features.

In [0]:
patients_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/workspace/default/raw_synthea/patients.csv")
)

display(patients_df)

In [0]:
patients_df.printSchema()

In [0]:
from pyspark.sql.functions import col, sum

print("Total rows:", patients_df.count())
print("Unique patient IDs:", patients_df.select("Id").distinct().count())

patients_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in patients_df.columns
]).display()

In [0]:
patients_df.select(
    "HEALTHCARE_EXPENSES",
    "HEALTHCARE_COVERAGE"
).describe().display()

In [0]:
silver_patients_df = patients_df.select(
    "Id",
    "BIRTHDATE",
    "DEATHDATE",
    "MARITAL",
    "RACE",
    "ETHNICITY",
    "GENDER",
    "CITY",
    "STATE",
    "COUNTY",
    "HEALTHCARE_EXPENSES",
    "HEALTHCARE_COVERAGE"
)

display(silver_patients_df)

In [0]:
from pyspark.sql.functions import col, floor, datediff, lit

silver_patients_df = silver_patients_df.withColumn(
    "AGE",
    floor(
        datediff(lit("2020-04-28"), col("BIRTHDATE")) / 365.25
    )
)

display(silver_patients_df)

In [0]:
silver_patients_df.select("AGE").describe().display()

In [0]:
from pyspark.sql.functions import when

silver_patients_df = silver_patients_df.withColumn(
    "AGE_GROUP",
    when(col("AGE") < 18, "0-17")
    .when(col("AGE") < 35, "18-34")
    .when(col("AGE") < 50, "35-49")
    .when(col("AGE") < 65, "50-64")
    .otherwise("65+")
)

display(silver_patients_df)

In [0]:
silver_patients_df.groupBy("AGE_GROUP").count().orderBy("AGE_GROUP").display()

In [0]:
silver_patients_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_patients")

In [0]:
%sql
SELECT *
FROM workspace.default.silver_patients
LIMIT 10;

## 2. Silver Layer — Encounters

Loads encounter-level healthcare activity, validates data quality and table relationships, derives financial and date features, and prepares encounter data for downstream utilization and cost analysis.


In [0]:
encounters_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/workspace/default/raw_synthea/encounters.csv")
)

display(encounters_df)

In [0]:
encounters_df.printSchema()

In [0]:
from pyspark.sql.functions import max

encounters_df.select(
    max("START").alias("LATEST_ENCOUNTER")
).display()

In [0]:
print("Total rows:", encounters_df.count())
print("Unique encounter IDs:", encounters_df.select("Id").distinct().count())

encounters_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in encounters_df.columns
]).display()

In [0]:
encounters_df.select(
    "BASE_ENCOUNTER_COST",
    "TOTAL_CLAIM_COST",
    "PAYER_COVERAGE"
).describe().display()

In [0]:
encounters_df.filter(
    col("BASE_ENCOUNTER_COST") != col("TOTAL_CLAIM_COST")
).count()

In [0]:
silver_encounters_df = encounters_df.select(
    "Id",
    "START",
    "STOP",
    "PATIENT",
    "ORGANIZATION",
    "PROVIDER",
    "PAYER",
    "ENCOUNTERCLASS",
    "CODE",
    "DESCRIPTION",
    "TOTAL_CLAIM_COST",
    "PAYER_COVERAGE",
    "REASONCODE",
    "REASONDESCRIPTION"
)

display(silver_encounters_df)

In [0]:
silver_encounters_df = silver_encounters_df.withColumn(
    "PATIENT_RESPONSIBILITY",
    col("TOTAL_CLAIM_COST") - col("PAYER_COVERAGE")
)

display(silver_encounters_df)

In [0]:
silver_encounters_df.select(
    "PATIENT_RESPONSIBILITY"
).describe().display()

In [0]:
silver_encounters_df = silver_encounters_df.withColumn(
    "ENCOUNTER_DURATION_MINUTES",
    (col("STOP").cast("long") - col("START").cast("long")) / 60
)

display(silver_encounters_df)

In [0]:
silver_encounters_df.select(
    "ENCOUNTER_DURATION_MINUTES"
).describe().display()

In [0]:
silver_encounters_df.select(
    "Id",
    "START",
    "STOP",
    "ENCOUNTERCLASS",
    "DESCRIPTION",
    "ENCOUNTER_DURATION_MINUTES"
).orderBy(
    col("ENCOUNTER_DURATION_MINUTES").desc()
).show(10, truncate=False)

In [0]:
silver_encounters_df = silver_encounters_df.drop(
    "ENCOUNTER_DURATION_MINUTES"
)

In [0]:
from pyspark.sql.functions import year, month, to_date

silver_encounters_df = (
    silver_encounters_df
    .withColumn("ENCOUNTER_DATE", to_date(col("START")))
    .withColumn("ENCOUNTER_YEAR", year(col("START")))
    .withColumn("ENCOUNTER_MONTH", month(col("START")))
)

display(silver_encounters_df)

In [0]:
silver_encounters_df.select(
    "ENCOUNTER_YEAR"
).describe().display()

In [0]:
silver_encounters_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_encounters")

In [0]:
%sql
SELECT *
FROM workspace.default.silver_encounters
LIMIT 10;

## 3. Silver Layer — Organizations

Loads healthcare organization data, validates record quality, and retains organization attributes relevant to utilization, revenue, and geographic analysis.

In [0]:
organizations_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/workspace/default/raw_synthea/organizations.csv")
)

display(organizations_df)

In [0]:
organizations_df.printSchema()

In [0]:
print("Total rows:", organizations_df.count())
print("Unique IDs:", organizations_df.select("Id").distinct().count())

organizations_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in organizations_df.columns
]).display()

In [0]:
silver_organizations_df = organizations_df.select(
    "Id",
    "NAME",
    "CITY",
    "STATE",
    "ZIP",
    "REVENUE",
    "UTILIZATION"
)

display(silver_organizations_df)

In [0]:
silver_organizations_df.select(
    "REVENUE",
    "UTILIZATION"
).describe().display()

In [0]:
silver_organizations_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_organizations")

In [0]:
%sql
SELECT *
FROM workspace.default.silver_organizations
LIMIT 10;

## 4. Silver Layer — Providers

Loads provider data, validates record quality, and retains provider attributes relevant to specialty, organization, location, and utilization analysis.

In [0]:
providers_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/workspace/default/raw_synthea/providers.csv")
)

display(providers_df)

In [0]:
providers_df.printSchema()

In [0]:
print("Total rows:", providers_df.count())
print("Unique IDs:", providers_df.select("Id").distinct().count())

providers_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in providers_df.columns
]).display()

In [0]:
silver_providers_df = providers_df.select(
    "Id",
    "ORGANIZATION",
    "NAME",
    "GENDER",
    "SPECIALITY",
    "CITY",
    "STATE",
    "UTILIZATION"
)

display(silver_providers_df)

In [0]:
silver_providers_df.groupBy(
    "SPECIALITY"
).count().orderBy(
    col("count").desc()
).display()

In [0]:
silver_providers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_providers")

In [0]:
%sql
SELECT *
FROM workspace.default.silver_providers
LIMIT 10;

## 5. Silver Layer — Payers

Loads payer data, validates financial and coverage fields, and retains attributes relevant to insurance coverage, encounter utilization, and financial analysis.

In [0]:
payers_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/workspace/default/raw_synthea/payers.csv")
)

display(payers_df)

In [0]:
payers_df.printSchema()

In [0]:
print("Total rows:", payers_df.count())
print("Unique IDs:", payers_df.select("Id").distinct().count())

payers_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in payers_df.columns
]).display()

In [0]:
silver_payers_df = payers_df.select(
    "Id",
    "NAME",
    "AMOUNT_COVERED",
    "AMOUNT_UNCOVERED",
    "REVENUE",
    "COVERED_ENCOUNTERS",
    "UNCOVERED_ENCOUNTERS",
    "COVERED_PROCEDURES",
    "UNCOVERED_PROCEDURES",
    "UNIQUE_CUSTOMERS",
    "QOLS_AVG",
    "MEMBER_MONTHS"
)

display(silver_payers_df)

In [0]:
silver_payers_df.select(
    "AMOUNT_COVERED",
    "AMOUNT_UNCOVERED",
    "REVENUE",
    "COVERED_ENCOUNTERS",
    "UNCOVERED_ENCOUNTERS",
    "UNIQUE_CUSTOMERS"
).describe().display()

In [0]:
silver_payers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_payers")

In [0]:
%sql
SELECT *
FROM workspace.default.silver_payers
LIMIT 10;

## 6. Silver Layer — Conditions

Loads patient condition records, validates completeness and duplicate records, and prepares condition data for analysis of condition prevalence and associated healthcare utilization.

In [0]:
conditions_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/workspace/default/raw_synthea/conditions.csv")
)

display(conditions_df)

In [0]:
conditions_df.printSchema()

In [0]:
print("Total rows:", conditions_df.count())

conditions_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in conditions_df.columns
]).display()

In [0]:
print("Total rows:", conditions_df.count())
print("Distinct rows:", conditions_df.distinct().count())

In [0]:
silver_conditions_df = conditions_df.select(
    "START",
    "STOP",
    "PATIENT",
    "ENCOUNTER",
    "CODE",
    "DESCRIPTION"
)

display(silver_conditions_df)

In [0]:
silver_conditions_df.groupBy(
    "DESCRIPTION"
).count().orderBy(
    col("count").desc()
).display()

In [0]:
silver_conditions_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_conditions")

In [0]:
%sql
SELECT *
FROM workspace.default.silver_conditions
LIMIT 10;

## 7. Silver Layer — Procedures

Loads procedure-level healthcare data, validates completeness, duplicates, and procedure costs, and prepares procedure records for downstream volume and cost analysis.

In [0]:
procedures_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/workspace/default/raw_synthea/procedures.csv")
)

display(procedures_df)

In [0]:
procedures_df.printSchema()

In [0]:
print("Total rows:", procedures_df.count())

procedures_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in procedures_df.columns
]).display()

In [0]:
print("Total rows:", procedures_df.count())
print("Distinct rows:", procedures_df.distinct().count())

In [0]:
procedures_df.select(
    "BASE_COST"
).describe().display()

In [0]:
silver_procedures_df = procedures_df.select(
    "DATE",
    "PATIENT",
    "ENCOUNTER",
    "CODE",
    "DESCRIPTION",
    "BASE_COST",
    "REASONCODE",
    "REASONDESCRIPTION"
)

display(silver_procedures_df)

In [0]:
silver_procedures_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_procedures")

In [0]:
%sql
SELECT *
FROM workspace.default.silver_procedures
LIMIT 10;

## 8. Data Relationship Validation

Validates referential integrity across the Silver layer before building analytical Gold tables. These checks confirm that encounter records successfully map to patients, organizations, providers, payers, conditions, and procedures.

In [0]:
%sql
SELECT COUNT(*) AS unmatched_encounters
FROM workspace.default.silver_encounters e
LEFT JOIN workspace.default.silver_patients p
    ON e.PATIENT = p.Id
WHERE p.Id IS NULL;

In [0]:
%sql
SELECT
    SUM(CASE WHEN o.Id IS NULL THEN 1 ELSE 0 END) AS unmatched_organizations,
    SUM(CASE WHEN pr.Id IS NULL THEN 1 ELSE 0 END) AS unmatched_providers,
    SUM(CASE WHEN py.Id IS NULL THEN 1 ELSE 0 END) AS unmatched_payers
FROM workspace.default.silver_encounters e
LEFT JOIN workspace.default.silver_organizations o
    ON e.ORGANIZATION = o.Id
LEFT JOIN workspace.default.silver_providers pr
    ON e.PROVIDER = pr.Id
LEFT JOIN workspace.default.silver_payers py
    ON e.PAYER = py.Id;

In [0]:
%sql
SELECT
    (SELECT COUNT(*)
     FROM workspace.default.silver_conditions c
     LEFT JOIN workspace.default.silver_encounters e
       ON c.ENCOUNTER = e.Id
     WHERE e.Id IS NULL) AS unmatched_conditions,

    (SELECT COUNT(*)
     FROM workspace.default.silver_procedures p
     LEFT JOIN workspace.default.silver_encounters e
       ON p.ENCOUNTER = e.Id
     WHERE e.Id IS NULL) AS unmatched_procedures;

## 9. Gold Layer — Business Analytics

Transforms validated Silver data into business-focused analytical tables for Power BI. The Gold layer focuses on healthcare utilization, financial performance, patient demographics, organizations, payers, conditions, and procedures.

**Analysis Period:** 2010–2019  
The analysis uses complete calendar years from 2010–2019. Earlier years contain substantially fewer encounter records, while 2020 is only a partial year ending April 28.

## Gold Table #1
## How is healthcare utilization and cost changing over time?

In [0]:
%sql
SELECT
    ENCOUNTER_YEAR AS YEAR,
    COUNT(*) AS ENCOUNTERS_PER_YEAR
FROM workspace.default.silver_encounters
GROUP BY ENCOUNTER_YEAR
ORDER BY ENCOUNTERS_PER_YEAR DESC;

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.default.gold_encounter_trends AS

SELECT
    ENCOUNTER_YEAR AS YEAR,
    ENCOUNTER_MONTH AS MONTH,
    COUNT(*) AS TOTAL_ENCOUNTERS,
    COUNT(DISTINCT PATIENT) AS UNIQUE_PATIENTS,
    SUM(TOTAL_CLAIM_COST) AS TOTAL_CLAIM_COST,
    SUM(PAYER_COVERAGE) AS TOTAL_PAYER_COVERAGE,
    SUM(PATIENT_RESPONSIBILITY) AS TOTAL_PATIENT_RESPONSIBILITY,
    AVG(TOTAL_CLAIM_COST) AS AVG_CLAIM_COST
FROM workspace.default.silver_encounters
WHERE ENCOUNTER_YEAR BETWEEN 2010 AND 2019
GROUP BY ENCOUNTER_YEAR, ENCOUNTER_MONTH;

In [0]:
%sql

SELECT *
FROM workspace.default.gold_encounter_trends
ORDER BY YEAR, MONTH;

## Gold Table #2
## Which types of healthcare encounters drive the most utilization and cost?

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.default.gold_encounter_types AS

SELECT
    ENCOUNTERCLASS AS ENCOUNTER_TYPE,
    COUNT(*) AS TOTAL_ENCOUNTERS,
    COUNT(DISTINCT PATIENT) AS UNIQUE_PATIENTS,
    SUM(TOTAL_CLAIM_COST) AS TOTAL_CLAIM_COST,
    AVG(TOTAL_CLAIM_COST) AS AVG_CLAIM_COST,
    SUM(PAYER_COVERAGE) AS TOTAL_PAYER_COVERAGE,
    SUM(PATIENT_RESPONSIBILITY) AS TOTAL_PATIENT_RESPONSIBILITY
FROM workspace.default.silver_encounters
WHERE ENCOUNTER_YEAR BETWEEN 2010 AND 2019
GROUP BY ENCOUNTERCLASS;

In [0]:
%sql

SELECT *
FROM workspace.default.gold_encounter_types
ORDER BY TOTAL_ENCOUNTERS DESC;

## Gold Table #3: Patient demographics 
## Which patient demographic groups have the highest healthcare utilization and costs?

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.default.gold_patient_demographics AS

SELECT
    p.AGE_GROUP,
    COUNT(*) AS TOTAL_ENCOUNTERS,
    COUNT(DISTINCT e.PATIENT) AS UNIQUE_PATIENTS,
    SUM(e.TOTAL_CLAIM_COST) AS TOTAL_CLAIM_COST,
    SUM(e.PAYER_COVERAGE) AS TOTAL_PAYER_COVERAGE,
    SUM(e.PATIENT_RESPONSIBILITY) AS TOTAL_PATIENT_RESPONSIBILITY
FROM workspace.default.silver_encounters AS e
INNER JOIN workspace.default.silver_patients AS p
    ON e.PATIENT = p.Id
WHERE e.ENCOUNTER_YEAR BETWEEN 2010 AND 2019
GROUP BY p.AGE_GROUP;

In [0]:
%sql

SELECT *
FROM workspace.default.gold_patient_demographics
ORDER BY TOTAL_ENCOUNTERS DESC;

## Gold Table #4: Organization Performance
## Which healthcare organizations handle the most encounters, and what costs are associated with that utilization?

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.default.gold_organization_performance AS

SELECT
    o.NAME AS ORGANIZATION_NAME,
    o.CITY AS CITY,
    COUNT(*) AS TOTAL_ENCOUNTERS,
    COUNT(DISTINCT e.PATIENT) AS UNIQUE_PATIENTS,
    SUM(e.TOTAL_CLAIM_COST) AS TOTAL_CLAIM_COST,
    SUM(e.PAYER_COVERAGE) AS TOTAL_PAYER_COVERAGE,
    SUM(e.PATIENT_RESPONSIBILITY) AS TOTAL_PATIENT_RESPONSIBILITY
FROM workspace.default.silver_encounters AS e
INNER JOIN workspace.default.silver_organizations AS o
    ON e.ORGANIZATION = o.Id
WHERE e.ENCOUNTER_YEAR BETWEEN 2010 AND 2019
GROUP BY o.NAME, o.CITY;

In [0]:
%sql

SELECT *
FROM workspace.default.gold_organization_performance
ORDER BY TOTAL_ENCOUNTERS DESC;

## Gold Table #5: Payer Performance
## How do insurance payers differ in encounter volume, coverage, and patient financial responsibility?

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.default.gold_payer_performance AS

SELECT
    p.NAME AS PAYER_NAME,
    COUNT(*) AS TOTAL_ENCOUNTERS,
    COUNT(DISTINCT e.PATIENT) AS UNIQUE_PATIENTS,
    SUM(e.TOTAL_CLAIM_COST) AS TOTAL_CLAIM_COST,
    SUM(e.PAYER_COVERAGE) AS TOTAL_PAYER_COVERAGE,
    SUM(e.PATIENT_RESPONSIBILITY) AS TOTAL_PATIENT_RESPONSIBILITY
FROM workspace.default.silver_encounters AS e
INNER JOIN workspace.default.silver_payers AS p
    ON e.PAYER = p.Id
WHERE e.ENCOUNTER_YEAR BETWEEN 2010 AND 2019
GROUP BY p.NAME;

In [0]:
%sql

SELECT *
FROM workspace.default.gold_payer_performance
ORDER BY TOTAL_ENCOUNTERS DESC;

## Gold Table #6: Conditions Analysis
## Which medical conditions are most commonly associated with encounters during 2010–2019?

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.default.gold_condition_analysis AS

SELECT
    c.DESCRIPTION AS CONDITION,
    COUNT(*) AS TOTAL_CONDITION_RECORDS,
    COUNT(DISTINCT e.PATIENT) AS UNIQUE_PATIENTS,
    SUM(e.TOTAL_CLAIM_COST) AS TOTAL_ASSOCIATED_CLAIM_COST
FROM workspace.default.silver_conditions AS c
INNER JOIN workspace.default.silver_encounters AS e
    ON c.ENCOUNTER = e.Id
WHERE e.ENCOUNTER_YEAR BETWEEN 2010 AND 2019
GROUP BY c.DESCRIPTION;

In [0]:
%sql

SELECT *
FROM workspace.default.gold_condition_analysis
ORDER BY TOTAL_CONDITION_RECORDS DESC;

## Gold Table #7: Procedure Cost Analysis
## Which procedures are performed most often, and which procedures account for the greatest procedure costs?

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.default.gold_procedure_analysis AS

SELECT
    DESCRIPTION AS PROCEDURE,
    COUNT(*) AS TOTAL_PROCEDURES,
    COUNT(DISTINCT PATIENT) AS UNIQUE_PATIENTS,
    SUM(BASE_COST) AS TOTAL_PROCEDURE_COST,
    AVG(BASE_COST) AS AVG_PROCEDURE_COST
FROM workspace.default.silver_procedures
WHERE YEAR(DATE) BETWEEN 2010 AND 2019
GROUP BY DESCRIPTION;

In [0]:
%sql

SELECT *
FROM workspace.default.gold_procedure_analysis
ORDER BY TOTAL_PROCEDURES DESC;

## Gold #8: Executive KPIs
## What are the overall healthcare utilization and financial metrics for 2010–2019?

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.default.gold_executive_kpis AS

SELECT
    COUNT(*) AS TOTAL_ENCOUNTERS,
    COUNT(DISTINCT PATIENT) AS UNIQUE_PATIENTS,
    SUM(TOTAL_CLAIM_COST) AS TOTAL_CLAIM_COST,
    SUM(PAYER_COVERAGE) AS TOTAL_PAYER_COVERAGE,
    SUM(PATIENT_RESPONSIBILITY) AS TOTAL_PATIENT_RESPONSIBILITY
FROM workspace.default.silver_encounters
WHERE ENCOUNTER_YEAR BETWEEN 2010 AND 2019;

In [0]:
%sql

SELECT *
FROM workspace.default.gold_executive_kpis;